In [11]:
# -*- coding: utf-8 -*-
"""用性能评估阶段保存的最终回归器生成 SHAP/LIME CSV（不绘图）。

重要：本 notebook 不再重新训练分类器或重新随机划分样本。
它加载 reviewer3_threshold_results/threshold_{5,10,50}/final_model.joblib，
并以 independent_test_predictions.csv 中的 original_sample_index 还原
性能评估阶段使用的同一独立测试集。SHAP/LIME 解释的是回归器的原始预测分数；
该分数用于按训练阶段选定的分类阈值划分高/低风险，不应直接称为校准概率。

输出目录：E:\\NO-NC\\submission\\revise\\SHAP_LIME_results
"""
from pathlib import Path
import warnings
import json

import numpy as np
import pandas as pd
import joblib
import shap
from lime.lime_tabular import LimeTabularExplainer

from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")

DATA_DIR = Path(r"E:\NO-NC\normalization\revise")
RESULTS_DIR = DATA_DIR / "reviewer3_threshold_results"
OUTPUT_DIR = Path(r"E:\NO-NC\submission\revise\SHAP_LIME_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILES = {
    5: DATA_DIR / "extract_20261019_train_005.csv",
    10: DATA_DIR / "extract_20261019_train_010.csv",
    50: DATA_DIR / "extract_20261019_train_050.csv",
}
CATEGORICAL = ["Parent material", "Climate classification", "Land use", "Texture"]
RANDOM_STATE = 20261019
N_LIME_FEATURES = 10
N_LIME_SAMPLES = 3000
N_SHAP_BACKGROUND = 100

# The SelectKBest pipeline saved for 10 µg/L pickles this scorer by reference
# to __main__. Define it before joblib.load so the saved pipeline can be restored.
def mixed_mutual_info_scores(X, y):
    values = np.asarray(X, dtype=float)
    discrete = np.zeros(values.shape[1], dtype=bool)
    for j in range(values.shape[1]):
        col = values[:, j]
        col = col[np.isfinite(col)]
        discrete[j] = np.unique(col).size <= 2
    return mutual_info_classif(
        values, y, discrete_features=discrete, random_state=RANDOM_STATE
    )


def load_xy(csv_path):
    """Load the transposed learning CSV; preserve sample order for index matching."""
    raw = pd.read_csv(csv_path, header=None, na_values=[-9999, "-9999"], dtype=object)
    if raw.shape[0] < 2 or raw.shape[1] < 2:
        raise ValueError(f"Invalid input CSV shape {raw.shape}: {csv_path}")
    target_name = str(raw.iat[0, 0]).strip().lower()
    if target_name not in {"y", "target", "learning target", "label"}:
        raise ValueError(f"Expected Y/target in first cell; found {raw.iat[0, 0]!r}")
    y = pd.to_numeric(raw.iloc[0, 1:], errors="coerce").reset_index(drop=True)
    names = raw.iloc[1:, 0].astype(str).str.strip().tolist()
    if len(names) != len(set(names)):
        raise ValueError("Duplicate explanatory-variable names in CSV")
    X = raw.iloc[1:, 1:].apply(pd.to_numeric, errors="coerce").T.reset_index(drop=True)
    X.columns = names
    X = X.replace([np.inf, -np.inf], np.nan)
    valid = y.notna()
    X = X.loc[valid].reset_index(drop=True)
    y = y.loc[valid].astype(int).reset_index(drop=True)
    if not set(y.unique()).issubset({0, 1}):
        raise ValueError(f"Y must be binary 0/1; found {sorted(y.unique())}")
    return X, y


def output_feature_names_and_sources(preprocessor, raw_features):
    """Build transformed names/source mapping without sklearn's newer API.

    Older scikit-learn versions do not implement get_feature_names_out on
    SimpleImputer. Derive the output columns directly from the fitted
    ColumnTransformer, imputers, and OneHotEncoder categories instead.
    """
    names, sources = [], []
    for transformer_name, transformer, columns in preprocessor.transformers_:
        if transformer_name == "remainder" or transformer == "drop":
            continue
        columns = list(columns) if not isinstance(columns, str) else [columns]
        if not columns:
            continue
        if transformer == "passthrough":
            names.extend(f"{transformer_name}__{col}" for col in columns)
            sources.extend(columns)
            continue

        steps = transformer.named_steps if hasattr(transformer, "named_steps") else {}
        imputer = next(
            (step for step in steps.values() if hasattr(step, "statistics_")),
            transformer if hasattr(transformer, "statistics_") else None,
        )
        kept_columns = columns
        if imputer is not None:
            stats = np.asarray(imputer.statistics_, dtype=object)
            # Older SimpleImputer drops all-empty columns. Newer versions may
            # retain them when keep_empty_features=True.
            keep_empty = bool(getattr(imputer, "keep_empty_features", False))
            if len(stats) == len(columns) and not keep_empty:
                kept_columns = [
                    col for col, stat in zip(columns, stats)
                    if not pd.isna(stat)
                ]

        encoder = next(
            (step for step in steps.values() if hasattr(step, "categories_")),
            transformer if hasattr(transformer, "categories_") else None,
        )
        if encoder is not None:
            categories = list(encoder.categories_)
            if len(categories) != len(kept_columns):
                raise ValueError(
                    f"Cannot map fitted encoder columns for transformer "
                    f"{transformer_name!r}: {len(categories)} category arrays, "
                    f"{len(kept_columns)} input variables"
                )
            for feature, category_values in zip(kept_columns, categories):
                for value in category_values:
                    names.append(f"{transformer_name}__{feature}_{value}")
                    sources.append(feature)
        else:
            # Numeric imputation/scaling preserves one output column per input.
            names.extend(f"{transformer_name}__{col}" for col in kept_columns)
            sources.extend(kept_columns)

    expected_width = sum(
        len(cols) for name, _, cols in preprocessor.transformers_
        if name != "remainder" and not (hasattr(preprocessor, "_remainder") and name == "remainder")
    )
    # The output width can differ from the input width because one-hot
    # encoding expands categories and imputers can drop all-empty columns.
    # The caller validates the final mapped width against the transformed data.
    if len(names) != len(sources):
        raise RuntimeError("Internal transformed-feature mapping is inconsistent")
    return names, sources


def impute_raw_for_explanation(preprocessor, X_raw):
    """Fill raw input missing values with the already-fitted pipeline statistics."""
    result = X_raw.copy()
    for name, transformer, columns in preprocessor.transformers_:
        if name == "remainder" or transformer == "drop" or not columns:
            continue
        columns = list(columns) if not isinstance(columns, str) else [columns]
        if hasattr(transformer, "named_steps"):
            imputers = [step for step in transformer.named_steps.values()
                        if hasattr(step, "statistics_")]
            imputer = imputers[0] if imputers else None
        elif hasattr(transformer, "statistics_"):
            imputer = transformer
        else:
            imputer = None
        if imputer is not None:
            for col, fill_value in zip(columns, imputer.statistics_):
                result[col] = result[col].fillna(fill_value)
    return result


def compute_shap_values(estimator, model_matrix):
    """Compute exact TreeSHAP contributions for the fitted regression model.

    Use each library's native contribution implementation for XGBoost and
    LightGBM. This avoids incompatibilities between older SHAP releases and
    newer booster formats. For sklearn forests, use tree-path-dependent SHAP;
    it is additive to the forest prediction and does not need background data.
    Returns (feature contributions, per-row base value).
    """
    matrix = np.asarray(model_matrix)
    module = estimator.__class__.__module__.lower()

    if "xgboost" in module and hasattr(estimator, "get_booster"):
        import xgboost as xgb
        contrib = estimator.get_booster().predict(
            xgb.DMatrix(matrix.astype(np.float32)), pred_contribs=True
        )
        contrib = np.asarray(contrib)
        if contrib.ndim != 2 or contrib.shape[1] != matrix.shape[1] + 1:
            raise ValueError(f"Unexpected XGBoost contribution shape: {contrib.shape}")
        return contrib[:, :-1], contrib[:, -1]

    if "lightgbm" in module and hasattr(estimator, "booster_"):
        contrib = np.asarray(
            estimator.booster_.predict(matrix, pred_contrib=True)
        )
        if contrib.ndim != 2 or contrib.shape[1] != matrix.shape[1] + 1:
            raise ValueError(f"Unexpected LightGBM contribution shape: {contrib.shape}")
        return contrib[:, :-1], contrib[:, -1]

    explainer = shap.TreeExplainer(
        estimator, feature_perturbation="tree_path_dependent"
    )
    values = explainer.shap_values(matrix, check_additivity=True)
    if isinstance(values, list):
        values = values[0]
    values = np.asarray(values)
    if values.ndim == 3 and values.shape[-1] == 1:
        values = values[:, :, 0]
    if values.shape != (matrix.shape[0], matrix.shape[1]):
        raise ValueError(f"Unexpected SHAP shape: {values.shape}")
    base = np.asarray(explainer.expected_value, dtype=float).reshape(-1)
    if len(base) != 1:
        raise ValueError(f"Unexpected SHAP expected_value: {explainer.expected_value}")
    return values, np.full(matrix.shape[0], float(base[0]))

def normalize_shap_array(values, n_rows, n_features):
    if hasattr(values, "values"):
        values = values.values
    if isinstance(values, list):
        values = values[0]
    values = np.asarray(values)
    if values.ndim == 3 and values.shape[-1] == 1:
        values = values[:, :, 0]
    if values.shape != (n_rows, n_features):
        raise ValueError(
            f"Unexpected SHAP shape {values.shape}; expected {(n_rows, n_features)}"
        )
    return values


def select_representative_cases(y_test, scores, cutoff):
    y = np.asarray(y_test, dtype=int)
    scores = np.asarray(scores, dtype=float)
    predicted = (scores >= cutoff).astype(int)
    cases = {
        "low_predicted_risk": int(np.argmin(scores)),
        "intermediate_predicted_risk": int(np.argmin(np.abs(scores - cutoff / 2.0))),
        "high_predicted_risk": int(np.argmax(scores)),
    }
    for role, mask, choose_high in [
        ("true_positive", (y == 1) & (predicted == 1), True),
        ("false_negative", (y == 1) & (predicted == 0), True),
        ("false_positive", (y == 0) & (predicted == 1), True),
    ]:
        idx = np.flatnonzero(mask)
        if len(idx):
            cases[role] = int(idx[np.argmax(scores[idx])] if choose_high else idx[0])
    rows = []
    for role, i in cases.items():
        rows.append({
            "role": role,
            "test_row_index": int(i),
            "original_sample_index": int(i),
            "y_true": int(y[i]),
            "prediction_score": float(scores[i]),
            # Kept for compatibility with the existing plotting notebook.
            # It is a regression score, not a calibrated probability.
            "predicted_probability": float(scores[i]),
            "predicted_class": int(predicted[i]),
            "classification_threshold": float(cutoff),
        })
    return cases, pd.DataFrame(rows)


def run_threshold(threshold):
    data_path = DATA_FILES[threshold]
    result_dir = RESULTS_DIR / f"threshold_{threshold}"
    model_path = result_dir / "final_model.joblib"
    predictions_path = result_dir / "independent_test_predictions.csv"
    if not data_path.exists() or not model_path.exists() or not predictions_path.exists():
        raise FileNotFoundError(
            f"Required data/model/test predictions missing for threshold {threshold}: "
            f"{data_path}, {model_path}, {predictions_path}"
        )

    X, y = load_xy(data_path)
    saved = joblib.load(model_path)
    pipeline = saved["pipeline"]
    features = list(saved["features"])
    model_name = str(saved["model"])
    feature_set = str(saved["feature_set"])
    cutoff = float(saved["classification_threshold"])
    if not set(features).issubset(X.columns):
        missing = sorted(set(features) - set(X.columns))
        raise ValueError(f"Model input predictors absent from CSV: {missing}")

    test_predictions = pd.read_csv(predictions_path)
    required = {"original_sample_index", "y_true", "prediction_score", "classification_threshold"}
    if not required.issubset(test_predictions.columns):
        raise ValueError(f"Missing columns in {predictions_path}: {required - set(test_predictions.columns)}")
    test_idx = test_predictions["original_sample_index"].astype(int).to_numpy()
    if len(np.unique(test_idx)) != len(test_idx) or np.any(test_idx < 0) or np.any(test_idx >= len(y)):
        raise ValueError("Invalid or duplicate original_sample_index in test predictions")
    y_test = y.iloc[test_idx].reset_index(drop=True)
    saved_y = test_predictions["y_true"].astype(int).reset_index(drop=True)
    if not np.array_equal(y_test.to_numpy(), saved_y.to_numpy()):
        raise ValueError("Labels at saved test indices do not match the input learning CSV")
    if not np.allclose(test_predictions["classification_threshold"].to_numpy(float), cutoff):
        raise ValueError("Saved model cutoff and independent-test cutoff disagree")

    train_mask = np.ones(len(X), dtype=bool)
    train_mask[test_idx] = False
    X_train = X.loc[train_mask, features].reset_index(drop=True)
    X_test = X.iloc[test_idx][features].reset_index(drop=True)
    y_train = y.loc[train_mask].reset_index(drop=True)

    # Confirm the restored fitted pipeline reproduces the final-model test scores.
    scores = np.asarray(pipeline.predict(X_test), dtype=float).reshape(-1)
    expected_scores = test_predictions["prediction_score"].to_numpy(dtype=float)
    if not np.allclose(scores, expected_scores, rtol=1e-5, atol=1e-7, equal_nan=False):
        max_diff = float(np.max(np.abs(scores - expected_scores)))
        raise ValueError(
            f"Loaded model does not reproduce saved independent-test scores "
            f"(max |difference|={max_diff:.3g}); do not interpret mismatched explanations."
        )
    predicted = (scores >= cutoff).astype(int)

    prep = pipeline.named_steps["prep"]
    X_train_effective = impute_raw_for_explanation(prep, X_train)
    X_test_effective = impute_raw_for_explanation(prep, X_test)
    Z_train = prep.transform(X_train_effective)
    Z_test = prep.transform(X_test_effective)
    transformed_names, transformed_sources = output_feature_names_and_sources(prep, features)
    if "select" in pipeline.named_steps:
        selector = pipeline.named_steps["select"]
        Z_train = selector.transform(Z_train)
        Z_test = selector.transform(Z_test)
        support = selector.get_support()
        transformed_names = [n for n, keep in zip(transformed_names, support) if keep]
        transformed_sources = [n for n, keep in zip(transformed_sources, support) if keep]
    estimator = pipeline.named_steps["model"]
    if Z_test.shape[1] != len(transformed_sources):
        raise ValueError("Transformed feature names do not match fitted model input width")

    # Explain the same final fitted regressor and its raw prediction score.
    # Tree-path-dependent / native booster contributions avoid the known
    # interventional additivity failure in this environment (SHAP 0.39).
    shap_transformed, shap_base_values = compute_shap_values(estimator, Z_test)
    shap_residual = np.abs(shap_transformed.sum(axis=1) + shap_base_values - scores)
    max_shap_residual = float(np.max(shap_residual))
    if max_shap_residual > 1e-5:
        raise ValueError(
            "SHAP additivity check failed for the fitted regressor: "
            f"max |base + sum(SHAP) - prediction_score|={max_shap_residual:.3g}"
        )

    source_features = [f for f in features if f in set(transformed_sources)]
    shap_by_source = np.zeros((len(X_test), len(source_features)), dtype=float)
    for j, feature in enumerate(source_features):
        cols = [k for k, src in enumerate(transformed_sources) if src == feature]
        shap_by_source[:, j] = shap_transformed[:, cols].sum(axis=1)
    importance = pd.DataFrame({
        "feature": source_features,
        "mean_abs_shap": np.abs(shap_by_source).mean(axis=0),
        "mean_shap": shap_by_source.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    importance.insert(0, "rank", np.arange(1, len(importance) + 1))

    out_dir = OUTPUT_DIR / f"threshold_{threshold}"
    out_dir.mkdir(parents=True, exist_ok=True)
    importance.to_csv(out_dir / "global_shap_importance.csv", index=False, encoding="utf-8-sig")

    # Full independent-test SHAP distribution; aggregate encoded columns back to source predictors.
    full = pd.DataFrame({
        "test_row_index": test_idx,
        "original_sample_index": test_idx,
        "y_true": y_test.to_numpy(),
        "prediction_score": scores,
        "predicted_probability": scores,  # legacy name for existing plot code
        "predicted_class": predicted,
        "classification_threshold": cutoff,
    })
    for j, feature in enumerate(source_features):
        full[feature] = X_test_effective[feature].to_numpy(dtype=float)
        full[f"SHAP__{feature}"] = shap_by_source[:, j]
    full.to_csv(out_dir / "full_test_shap_beeswarm_data.csv", index=False, encoding="utf-8-sig")

    case_map, cases_df = select_representative_cases(y_test, scores, cutoff)
    cases_df.to_csv(out_dir / "representative_test_cases.csv", index=False, encoding="utf-8-sig")

    # LIME explains the same regressor's raw score. It uses original variables;
    # nominal predictors are treated as categorical and all missing values are
    # filled using the fitted preprocessing statistics before perturbation.
    categorical_features = [features.index(f) for f in CATEGORICAL if f in features]
    categorical_names = {}
    category_values = {}
    lime_train = X_train_effective.to_numpy(dtype=float).copy()
    lime_test = X_test_effective.to_numpy(dtype=float).copy()
    for i in categorical_features:
        # LIME represents categorical values as integer category IDs. Keep a
        # reversible map because the fitted pipeline expects original codes.
        vals = np.unique(np.concatenate([
            lime_train[:, i], lime_test[:, i]
        ])).astype(float)
        category_values[i] = vals
        code = {float(value): j for j, value in enumerate(vals)}
        categorical_names[i] = [str(value) for value in vals]
        lime_train[:, i] = np.asarray([code[float(v)] for v in lime_train[:, i]])
        lime_test[:, i] = np.asarray([code[float(v)] for v in lime_test[:, i]])

    def lime_predict(raw_lime_rows):
        decoded = np.asarray(raw_lime_rows, dtype=float).copy()
        for col, vals in category_values.items():
            ids = np.clip(np.rint(decoded[:, col]).astype(int), 0, len(vals) - 1)
            decoded[:, col] = vals[ids]
        return pipeline.predict(pd.DataFrame(decoded, columns=features))

    lime_explainer = LimeTabularExplainer(
        training_data=lime_train,
        feature_names=features,
        categorical_features=categorical_features,
        categorical_names=categorical_names,
        mode="regression",
        discretize_continuous=True,
        random_state=RANDOM_STATE,
    )

    local_rows = []
    for role, test_i in case_map.items():
        x_case = lime_test[test_i]
        lime_exp = lime_explainer.explain_instance(
            data_row=x_case,
            predict_fn=lime_predict,
            num_features=min(N_LIME_FEATURES, len(features)),
            num_samples=N_LIME_SAMPLES,
        )
        lime_map = dict(lime_exp.as_list())
        for j, feature in enumerate(source_features):
            value = float(X_test_effective.iloc[test_i][feature])
            shap_value = float(shap_by_source[test_i, j])
            local_rows.append({
                "role": role,
                "test_row_index": int(test_i),
                "original_sample_index": int(test_idx[test_i]),
                "y_true": int(y_test.iloc[test_i]),
                "predicted_class": int(predicted[test_i]),
                "classification_threshold": cutoff,
                "prediction_score": float(scores[test_i]),
                "predicted_probability": float(scores[test_i]),  # legacy alias
                "feature": feature,
                "feature_value": value,
                "shap_base_value": float(shap_base_values[test_i]),
                "shap_value_prediction": shap_value,
                "shap_value_class1": shap_value,  # legacy alias for the existing plot notebook
                "abs_shap_value": abs(shap_value),
                "lime_weight": np.nan,
            })
        for condition, weight in lime_map.items():
            local_rows.append({
                "role": role,
                "test_row_index": int(test_i),
                "original_sample_index": int(test_idx[test_i]),
                "y_true": int(y_test.iloc[test_i]),
                "predicted_class": int(predicted[test_i]),
                "classification_threshold": cutoff,
                "prediction_score": float(scores[test_i]),
                "predicted_probability": float(scores[test_i]),
                "feature": f"LIME condition: {condition}",
                "feature_value": np.nan,
                "shap_base_value": float(shap_base_values[test_i]),
                "shap_value_prediction": np.nan,
                "shap_value_class1": np.nan,
                "abs_shap_value": np.nan,
                "lime_weight": float(weight),
            })
    pd.DataFrame(local_rows).to_csv(
        out_dir / "local_SHAP_LIME_explanations.csv", index=False, encoding="utf-8-sig"
    )

    info = {
        "target_threshold_ug_L": threshold,
        "model": model_name,
        "feature_set": feature_set,
        "model_file": str(model_path),
        "test_predictions_file": str(predictions_path),
        "n_samples": int(len(X)),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "n_positive_test": int(y_test.sum()),
        "n_features_input": int(len(features)),
        "n_features_transformed_for_model": int(Z_test.shape[1]),
        "n_source_features_explained": int(len(source_features)),
        "classification_threshold": cutoff,
        "explained_output": "fitted regressor prediction score (not calibrated probability)",
        "selected_source_features_explained": "; ".join(source_features),
        "prediction_score_max_abs_difference_vs_saved_test": float(np.max(np.abs(scores - expected_scores))),
        "max_shap_additivity_residual": max_shap_residual,
    }
    pd.DataFrame([info]).to_csv(out_dir / "analysis_info.csv", index=False, encoding="utf-8-sig")
    print(
        f"{threshold} µg/L: {model_name} ({feature_set}); "
        f"n_test={len(X_test)}, features explained={len(source_features)}, "
        f"max score difference={info['prediction_score_max_abs_difference_vs_saved_test']:.3g}"
    )
    print(f"CSV outputs: {out_dir}")


def main():
    for threshold in (5, 10, 50):
        run_threshold(threshold)
    print(f"All SHAP/LIME CSV files saved under: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

5 µg/L: RandomForestRegressor (all); n_test=808, features explained=31, max score difference=9.71e-17
CSV outputs: E:\NO-NC\submission\revise\SHAP_LIME_results\threshold_5
10 µg/L: XGBoostRegressor (SelectKBest); n_test=808, features explained=19, max score difference=9.71e-17
CSV outputs: E:\NO-NC\submission\revise\SHAP_LIME_results\threshold_10
50 µg/L: LightGBMRegressor (correlation); n_test=808, features explained=28, max score difference=9.96e-17
CSV outputs: E:\NO-NC\submission\revise\SHAP_LIME_results\threshold_50
All SHAP/LIME CSV files saved under: E:\NO-NC\submission\revise\SHAP_LIME_results
